In [73]:
import torch
import pandas as pd
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    classification_report
)

data = pd.read_csv("../data/synthetic_data.csv")

print(data.head())
print(data.info())
print(data.describe())
print(data.shape)

class IrisNeuralNetwork:
    def __init__(self, input_size, hidden1_size, hidden2_size):

        # Input -> hidden layer 1
        self.w12 = torch.randn(input_size, hidden1_size)
        self.b1 = torch.zeros(hidden1_size)

        # Hidden layer 1 -> hidden layer 2
        self.w23 = torch.randn(hidden1_size, hidden2_size)
        self.b2 = torch.zeros(hidden2_size)

        # Hidden layer 2 -> output
        self.w34 = torch.randn(hidden2_size, 1)
        self.b3 = torch.zeros(1)

    def sigmoid(self, x):
        return 1 / (1 + torch.exp(-x))

    def forward(self, x):
        # Layer 1
        z1 = x @ self.w12 + self.b1
        a1 = self.sigmoid(z1)

        # Layer 2
        z2 = a1 @ self.w23 + self.b2
        a2 = self.sigmoid(z2)

        # Output layer
        z3 = a2 @ self.w34 + self.b3
        a3 = self.sigmoid(z3)

        return a3

history = {
    'train_loss': [],
    'test_loss': [],
    'test_accuracy': []
}

         X1        X2  output
0  0.994416  0.533942       1
1  1.177820 -1.126290       0
2 -0.945938 -0.848435       0
3  0.936008 -0.305729       1
4 -0.694159  0.698876       1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X1      1000 non-null   float64
 1   X2      1000 non-null   float64
 2   output  1000 non-null   int64  
dtypes: float64(2), int64(1)
memory usage: 23.6 KB
None
                X1           X2       output
count  1000.000000  1000.000000  1000.000000
mean     -0.027157    -0.041622     0.489000
std       0.881587     0.867416     0.500129
min      -1.499560    -1.493940     0.000000
25%      -0.776087    -0.756851     0.000000
50%      -0.031119    -0.085191     0.000000
75%       0.767149     0.699188     1.000000
max       1.496600     1.499670     1.000000
(1000, 3)


In [74]:
x = torch.tensor(
    data[["X1", "X2"]].values,
    dtype=torch.float32
)

y = torch.tensor(
    data["output"].values,
    dtype=torch.float32
).reshape(-1, 1)

print(x.shape, y.shape)

N = int(0.8 * len(x))

x_train, x_test = x[:N], x[N:]
y_train, y_test = y[:N], y[N:]

print(f"Training set: {x_train.shape}, {y_train.shape}")
print(f"Test set: {x_test.shape}, {y_test.shape}")

torch.Size([1000, 2]) torch.Size([1000, 1])
Training set: torch.Size([800, 2]), torch.Size([800, 1])
Test set: torch.Size([200, 2]), torch.Size([200, 1])


In [75]:
def binary_cross_entropy(y, prediction):

    # Prevent log(0)
    epsilon = 1e-7

    prediction = torch.clamp(
        prediction,
        epsilon,
        1 - epsilon
    )

    loss = -(
        y * torch.log(prediction)
        +
        (1 - y) * torch.log(1 - prediction)
    )

    return loss.mean()

def backward(model, x, y, prediction):

    # Number of samples
    m = x.shape[0]

    # -----------------------------------------
    # Recalculate forward-pass intermediates
    # -----------------------------------------

    z1 = x @ model.w12 + model.b1
    a1 = model.sigmoid(z1)

    z2 = a1 @ model.w23 + model.b2
    a2 = model.sigmoid(z2)

    z3 = a2 @ model.w34 + model.b3
    a3 = prediction

    # -----------------------------------------
    # Output layer
    # -----------------------------------------

    # For sigmoid + binary cross entropy:
    # dL/dz3 = a3 - y
    dz3 = a3 - y

    dw34 = (a2.T @ dz3) / m
    db3 = dz3.mean(dim=0)

    # -----------------------------------------
    # Hidden layer 2
    # -----------------------------------------

    da2 = dz3 @ model.w34.T

    dz2 = da2 * a2 * (1 - a2)

    dw23 = (a1.T @ dz2) / m
    db2 = dz2.mean(dim=0)

    # -----------------------------------------
    # Hidden layer 1
    # -----------------------------------------

    da1 = dz2 @ model.w23.T

    dz1 = da1 * a1 * (1 - a1)

    dw12 = (x.T @ dz1) / m
    db1 = dz1.mean(dim=0)

    return dw12, db1, dw23, db2, dw34, db3


def update_parameters(model, gradients, learning_rate, count):

    dw12, db1, dw23, db2, dw34, db3 = gradients

    model.w12 -= learning_rate * dw12
    model.b1  -= learning_rate * db1

    model.w23 -= learning_rate * dw23
    model.b2  -= learning_rate * db2

    model.w34 -= learning_rate * dw34
    model.b3  -= learning_rate * db3

    # if count % 100 == 0:
    #     print("Updated parameters:\n")
    #     print(f"  W12: {model.w12.shape}")
    #     print(f"  B1: {model.b1.shape}\n")
    #     print(f"  W23: {model.w23.shape}")
    #     print(f"  B2: {model.b2.shape}\n")
    #     print(f"  W34: {model.w34.shape}")
    #     print(f"  B3: {model.b3.shape}\n")


In [81]:
def train(model, x_train, y_train, epochs, learning_rate, count):

    for epoch in range(epochs):
        count+=1    

        # Forward pass
        prediction = model.forward(x_train)

        # Calculate loss
        train_loss = binary_cross_entropy(
            y_train,
            prediction
        )

        # Backpropagation
        gradients = backward(
            model,
            x_train,
            y_train,
            prediction
        )

        # dw12, db1, dw23, db2, dw34, db3 = gradients

        # if epoch % 100 == 0:
        #     print(
        #         f"dw12={dw12.abs().mean().item():.8f}, "
        #         f"dw23={dw23.abs().mean().item():.8f}, "
        #         f"dw34={dw34.abs().mean().item():.8f}"
        #     )

        # Gradient descent
        update_parameters(
            model,
            gradients,
            learning_rate,
            count
        )

        history['train_loss'].append(train_loss.item())        

        if epoch % 100 == 0:
            print(
                f"Epoch {epoch:4d} | "
                f"Loss: {train_loss.item():.6f}"
            )


In [87]:
model = IrisNeuralNetwork(
    input_size=2,
    hidden1_size=3,
    hidden2_size=2
)


train(
    model,
    x_train,
    y_train,
    epochs=5000,
    learning_rate=0.1,
    count=0
)

Epoch    0 | Loss: 0.761523
Epoch  100 | Loss: 0.691791
Epoch  200 | Loss: 0.691681
Epoch  300 | Loss: 0.691572
Epoch  400 | Loss: 0.691460
Epoch  500 | Loss: 0.691343
Epoch  600 | Loss: 0.691220
Epoch  700 | Loss: 0.691090
Epoch  800 | Loss: 0.690951
Epoch  900 | Loss: 0.690803
Epoch 1000 | Loss: 0.690644
Epoch 1100 | Loss: 0.690472
Epoch 1200 | Loss: 0.690286
Epoch 1300 | Loss: 0.690084
Epoch 1400 | Loss: 0.689863
Epoch 1500 | Loss: 0.689623
Epoch 1600 | Loss: 0.689359
Epoch 1700 | Loss: 0.689071
Epoch 1800 | Loss: 0.688754
Epoch 1900 | Loss: 0.688406
Epoch 2000 | Loss: 0.688023
Epoch 2100 | Loss: 0.687602
Epoch 2200 | Loss: 0.687139
Epoch 2300 | Loss: 0.686629
Epoch 2400 | Loss: 0.686068
Epoch 2500 | Loss: 0.685450
Epoch 2600 | Loss: 0.684772
Epoch 2700 | Loss: 0.684027
Epoch 2800 | Loss: 0.683211
Epoch 2900 | Loss: 0.682318
Epoch 3000 | Loss: 0.681343
Epoch 3100 | Loss: 0.680281
Epoch 3200 | Loss: 0.679128
Epoch 3300 | Loss: 0.677878
Epoch 3400 | Loss: 0.676529
Epoch 3500 | Loss: 0

In [88]:
# -----------------------------------------
# Test
# -----------------------------------------

def evaluate(model, x_test, y_test):
    with torch.no_grad():

        probabilities = model.forward(x_test)

        loss = binary_cross_entropy(
            y_test,
            probabilities)

        predictions = (probabilities >= 0.5).float()

        accuracy = (
            predictions == y_test
        ).float().mean()

        print()
        print("Test results:")
        print(
            f"Overall score: {accuracy.item():.7f} | ",
            f"Test Loss: {loss:.6f}"
            )

        print()

        for i in range(len(y_test)):

            actual = int(y_test[i].item())
            predicted = int(predictions[i].item())
            probability = probabilities[i].item()

            print(
                f"{i:03d}: "
                f"actual: {actual} "
                f"predict: {predicted} "
                f"prob: {probability:.7f}"
            )

        return loss.item(), accuracy.item(), predictions, probabilities


test_loss, accuracy, predictions, probabilities = evaluate(
    model, 
    x_test,
    y_test
)

history['test_loss'].append(test_loss)
history['test_accuracy'].append(accuracy)



Test results:
Overall score: 0.6450000 |  Test Loss: 0.644987

000: actual: 0 predict: 1 prob: 0.5466505
001: actual: 1 predict: 0 prob: 0.4056046
002: actual: 0 predict: 0 prob: 0.4798906
003: actual: 1 predict: 1 prob: 0.5245988
004: actual: 1 predict: 0 prob: 0.4957747
005: actual: 0 predict: 0 prob: 0.2767745
006: actual: 0 predict: 0 prob: 0.2891803
007: actual: 0 predict: 1 prob: 0.6152402
008: actual: 1 predict: 1 prob: 0.5088987
009: actual: 0 predict: 0 prob: 0.3244815
010: actual: 0 predict: 1 prob: 0.5533313
011: actual: 0 predict: 1 prob: 0.5398837
012: actual: 1 predict: 1 prob: 0.5355852
013: actual: 0 predict: 0 prob: 0.4696176
014: actual: 1 predict: 1 prob: 0.5359936
015: actual: 1 predict: 1 prob: 0.5050857
016: actual: 1 predict: 1 prob: 0.5823808
017: actual: 1 predict: 0 prob: 0.4533279
018: actual: 0 predict: 1 prob: 0.6076059
019: actual: 1 predict: 1 prob: 0.5232793
020: actual: 1 predict: 0 prob: 0.3656676
021: actual: 1 predict: 0 prob: 0.4586668
022: actual:

In [89]:
accuracy = accuracy_score(
    y_test.numpy(),
    predictions.numpy()
)

precision = precision_score(
    y_test.numpy(),
    predictions.numpy()
)

recall = recall_score(
    y_test.numpy(),
    predictions.numpy()
)

f1 = f1_score(
    y_test.numpy(),
    predictions.numpy()
)

cm = confusion_matrix(
    y_test.numpy(),
    predictions.numpy()
)

report = classification_report(
    y_test.numpy(),
    predictions.numpy(),
    output_dict=True
)

print("Confusion Matrix:")
print(cm)
print("Classification Report:")
print(report)

Confusion Matrix:
[[56 40]
 [31 73]]
Classification Report:
{'0.0': {'precision': 0.6436781609195402, 'recall': 0.5833333333333334, 'f1-score': 0.6120218579234973, 'support': 96.0}, '1.0': {'precision': 0.6460176991150443, 'recall': 0.7019230769230769, 'f1-score': 0.6728110599078341, 'support': 104.0}, 'accuracy': 0.645, 'macro avg': {'precision': 0.6448479300172922, 'recall': 0.6426282051282051, 'f1-score': 0.6424164589156657, 'support': 200.0}, 'weighted avg': {'precision': 0.6448947207812022, 'recall': 0.645, 'f1-score': 0.6436322429553525, 'support': 200.0}}


In [90]:
experiment = {
    'name': 'Random Initialization',
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,                 
    'history': history,
    'classification_report': report,
    'confusion_matrix': cm
}

In [91]:
import joblib
joblib.dump(
    experiment,
    "../dictionary/random_initialization_experiment.pkl"
)

torch.save(model, "../models/random_weights.pt")

In [92]:
print("Class 0:", (y_test == 0).sum().item())
print("Class 1:", (y_test == 1).sum().item())

print("\n\n")

print("Predicted 0:", (predictions == 0).sum().item())
print("Predicted 1:", (predictions == 1).sum().item())

Class 0: 96
Class 1: 104



Predicted 0: 87
Predicted 1: 113


In [93]:
print(probabilities[:20].flatten())

tensor([0.5467, 0.4056, 0.4799, 0.5246, 0.4958, 0.2768, 0.2892, 0.6152, 0.5089,
        0.3245, 0.5533, 0.5399, 0.5356, 0.4696, 0.5360, 0.5051, 0.5824, 0.4533,
        0.6076, 0.5233])


In [94]:
print(x_train.shape, y_train.shape)

torch.Size([800, 2]) torch.Size([800, 1])
